# GRPO Loss 数值行为分析

## 要回答的三个问题

1. **为什么 GRPO 的 loss 经常是负数？** 训练日志里 loss = -0.03 是出 bug 了吗？
2. **为什么 loss 会「上升」？** 模型明明在变好，loss 却在涨，是不是训练发散了？
3. **loss 的绝对值能反映训练进展吗？** 该不该盯着 loss 曲线调参？

## 一句话结论

**GRPO 的 loss 值本身几乎不携带训练进展的信息，不要盯着它看。**

原因藏在优势函数的定义里——它是 z-score，所以：

$$\sum_{i=1}^{G} \hat{A}_i = \sum_{i=1}^{G} \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})} = \frac{1}{\text{std}(\mathbf{r})}\left(\sum_i r_i - G \cdot \text{mean}(\mathbf{r})\right) \equiv 0$$

**组内优势之和恒等于 0**，与奖励具体取什么值完全无关。这是理解下面所有现象的总钥匙。

下面逐个拆解，并且会发现一个反直觉的事实：
**本 notebook 早期版本画出的「loss 随奖励上升」曲线，其实是 float32 的舍入噪声。**

In [ ]:
import torch
import matplotlib.pyplot as plt

---
## 0. 基础组件

沿用 `grpo_loss.ipynb` 中已验证的实现。

> ⚠️ 注意 KL 的写法：早期版本写作 `(ref.exp() / pi.exp())`，数学上等价于 `exp(ref - pi)`，
> 但它要求 `pi.exp()` 和 `ref.exp()` 各自可表示。单 token 的 log prob 通常在 $-15$ 以上没问题，
> 而**序列级 log prob**（整条响应累加，GSPO 正是这么做的）轻易能到 $-100$ 以下：
> float32 在 $-100$ 附近进入次正规数、精度已损失约 13%，到 $-120$ 直接归零，相除得到 `nan`。
> 正确做法是在 log 空间先做减法再 exp —— 只对 `ref - pi` 这个小量取指数。

In [ ]:
def grpo_kl(pi_logprob, pi_ref_logprob):
    """
    k3 KL 估计量：r - log(r) - 1，其中 r = pi_ref / pi_theta

    在 log 空间实现（先减后 exp），避免 exp 下溢。
    性质：恒 >= 0，pi_theta == pi_ref 时为 0。
    """
    log_r = pi_ref_logprob - pi_logprob
    return torch.exp(log_r) - log_r - 1.0


def grpo_advantage(rewards, eps=1e-5):
    """组相对优势（z-score）。关键性质：sum(A) 恒为 0。"""
    return (rewards - rewards.mean()) / (rewards.std() + eps)


# ══════════ 数值下溢对比 ══════════
# 单 token 的 log prob 一般在 -15 以上，不会出问题；
# 但序列级 log prob（整条响应累加，GSPO 就是这么做的）轻易能到 -100 以下。
print("早期写法 (ref.exp() / pi.exp())  vs  修正写法 exp(ref - pi)")
print(f"{'log prob':>10} | {'pi.exp()':>12} | {'早期写法':>12} | {'修正写法':>12} | 状态")
print("-" * 70)
for v in [-30.0, -85.0, -100.0, -120.0]:
    pi, ref = torch.tensor(v), torch.tensor(v - 0.5)
    naive = ((ref.exp() / pi.exp()) - (ref - pi) - 1).item()
    safe = grpo_kl(pi, ref).item()
    if naive != naive:                       # nan 检测
        status = "❌ 完全下溢 → nan"
    elif abs(naive - safe) > 1e-4:
        status = f"⚠️ 次正规数，误差 {abs(naive-safe)/safe:.0%}"
    else:
        status = "✅ 尚且正常"
    print(f"{v:>10.0f} | {pi.exp().item():>12.3e} | {naive:>12.6f} | {safe:>12.6f} | {status}")

print("\n→ 修正写法在全部量级下都返回正确值 0.106531，因为它只对 (ref - pi) = -0.5 取 exp，")
print("  而早期写法要先分别算 exp(-120) —— 已经归零，再相除得到 0/0 = nan。")

# ══════════ 总钥匙：sum(A) 恒为 0 ══════════
print("\n无论奖励怎么变，组内优势之和恒为 0：")
for r in [torch.tensor([1.,0,0,0,0,0,0,0]),
          torch.tensor([1.,1,1,0,0,0,0,0]),
          torch.tensor([1.,1,1,1,1,1,1,0]),
          torch.tensor([3.,1,4,1,5,9,2,6])]:
    A = grpo_advantage(r)
    print(f"  奖励={str(r.tolist()):<40} sum(A) = {A.sum():+.3e}")

---
## 1. 为什么 loss 是负数？

**因为 GRPO 的 loss 不是交叉熵，它是一个「策略梯度替代目标」的相反数。**

交叉熵有下界 0，所以负值意味着出错。但 GRPO 的 loss 定义是：

$$\mathcal{L} = -\underbrace{\left[r \cdot \hat{A} - \beta \mathbb{D}_{\text{KL}}\right]}_{\text{要最大化的目标 } \mathcal{J}}$$

只要目标 $\mathcal{J} > 0$，loss 就是负的。而单个样本的 $\mathcal{J}$ 符号**几乎完全由 $\hat{A}_i$ 的符号决定**
（$r > 0$ 恒成立，$\beta$ 很小）：

- $\hat{A}_i > 0$（这条响应比同组平均好）→ $\mathcal{J} > 0$ → **loss 为负**
- $\hat{A}_i < 0$（比同组平均差）→ $\mathcal{J} < 0$ → **loss 为正**

由于 $\sum_i \hat{A}_i = 0$，**每一组里必然同时存在正负 loss**（除非全部为 0）。
这是设计使然，不是 bug。

In [ ]:
def minimal_grpo_loss(pi_logprob, pi_old_logprob, pi_ref_logprob, rewards,
                      beta=0.01, is_debug=True):
    """
    最简 GRPO loss（省略 min/clip，便于观察数值行为）

        loss_i = -( ratio * A_i - beta * KL )

    省略 clip 是因为本 notebook 关注 loss 的数值构成，而非信任域机制；
    完整实现见 grpo_loss.ipynb。
    """
    KL = grpo_kl(pi_logprob, pi_ref_logprob)
    A = grpo_advantage(rewards)
    ratio = torch.exp(pi_logprob - pi_old_logprob)
    loss = -(ratio * A - beta * KL)
    if is_debug:
        print(f'[Rewards] {rewards.tolist()}')
        print(f'[Adv]     {[round(v, 4) for v in A.tolist()]}')
        print(f'[Loss]    {[round(v, 4) for v in loss.tolist()]}')
    return loss


# ══════════ loss 符号 vs 优势符号 ══════════
pi_logprob     = torch.tensor(0.5).log()
pi_old_logprob = torch.tensor(0.5).log()      # ratio = 1
pi_ref_logprob = torch.tensor(0.6).log()
rewards = torch.tensor([1., 0, 0, 0, 0, 0, 0, 0])

loss = minimal_grpo_loss(pi_logprob, pi_old_logprob, pi_ref_logprob, rewards)
A = grpo_advantage(rewards)

print("\n逐样本核对「优势符号 → loss 符号」：")
print(f"{'样本':>4} | {'奖励':>5} | {'优势 A':>9} | {'loss':>9} | 符号关系")
print("-" * 52)
for i in range(len(rewards)):
    rel = "A>0 → loss<0 ✅" if A[i] > 0 else "A<0 → loss>0 ✅"
    print(f"{i:>4} | {rewards[i]:>5.0f} | {A[i]:>+9.4f} | {loss[i]:>+9.4f} | {rel}")

print(f"\n负 loss 样本数: {(loss < 0).sum().item()} / {len(loss)}")
print("→ 只要组内有样本优于平均，就必然出现负 loss。这是正常现象。")

---
## 2. 为什么 loss 的**和**几乎恒定？

把组内 loss 求和，代入 $\sum_i \hat{A}_i = 0$：

$$\sum_{i=1}^{G} \mathcal{L}_i = -\sum_i \left(r_i \hat{A}_i - \beta \text{KL}_i\right)
= -\underbrace{\sum_i r_i \hat{A}_i}_{\text{关键项}} + \beta \sum_i \text{KL}_i$$

**在本 notebook 的玩具设定里，所有样本共用同一个标量 ratio $r$**，于是：

$$\sum_i r_i \hat{A}_i = r \sum_i \hat{A}_i = r \cdot 0 = 0$$

策略梯度项被完全消掉，只剩下 KL 项：

$$\boxed{\sum_i \mathcal{L}_i = \beta \sum_i \text{KL}_i = G \cdot \beta \cdot \text{KL}}$$

**这个值与奖励完全无关** —— 无论组内答对 1 题还是 7 题，loss 之和都一样。

In [ ]:
# ══════════ 验证：loss.sum() 恒等于 G * beta * KL，与奖励无关 ══════════
pi_logprob     = torch.tensor(0.4).log()
pi_old_logprob = torch.tensor(0.3).log()       # ratio = 4/3，标量，全组共用
pi_ref_logprob = torch.tensor(0.401).log()
beta, G = 0.01, 8

kl_val = grpo_kl(pi_logprob, pi_ref_logprob)
theory = G * beta * kl_val

print(f"KL = {kl_val:.6e}")
print(f"理论 loss.sum() = G·β·KL = {G}×{beta}×{kl_val:.3e} = {theory:.6e}\n")
print(f"{'答对题数':>8} | {'loss.sum() 实测':>18} | {'理论值':>14} | {'相对误差':>10}")
print("-" * 62)

rewards = torch.zeros(G)
for i in range(G):
    rewards[i] = 1.0
    l = minimal_grpo_loss(pi_logprob, pi_old_logprob, pi_ref_logprob,
                          rewards, beta=beta, is_debug=False).sum()
    rel_err = abs(l - theory) / theory
    print(f"{int(rewards.sum()):>8} | {l.item():>18.6e} | {theory.item():>14.6e} | {rel_err:>9.1%}")

print("\n→ 实测值在理论值附近上下跳动，且跳动幅度远大于理论值本身。")
print("  下一节说明这些跳动是什么。")

---
## 3. ⚠️ 那条「loss 随奖励上升」的曲线是**浮点噪声**

上一节实测值围绕理论值上下跳动，相对误差高达数百 %。
如果把这些点连成曲线，看起来会像一条「有趋势」的线——但它没有任何物理意义。

**根因是灾难性抵消（catastrophic cancellation）**：

逐样本 loss 的量级可以到 $\pm 50$，而它们的真实和只有 $\sim 4 \times 10^{-6}$。
两者差了 **7 个数量级**，而 float32 只有约 7 位十进制有效数字——
真实信号恰好被舍入误差完全淹没。

下面用三条证据确认。

In [ ]:
# ══════════ 证据 1：切到 float64，曲线立刻变平 ══════════
def loss_sum_curve(dtype, n=8, beta=0.01):
    """扫描答对题数，记录 loss.sum()"""
    pi  = torch.tensor(0.1,   dtype=dtype).log()
    old = torch.tensor(0.005, dtype=dtype).log()      # ratio = 20，放大抵消效应
    ref = torch.tensor(0.101, dtype=dtype).log()
    out, rewards = [], torch.zeros(n, dtype=dtype)
    for i in range(n):
        rewards[i] = 1.0
        out.append(minimal_grpo_loss(pi, old, ref, rewards,
                                     beta=beta, is_debug=False).sum().item())
    return out, (n * beta * grpo_kl(pi, ref)).item()

c32, t32 = loss_sum_curve(torch.float32)
c64, t64 = loss_sum_curve(torch.float64)

print(f"{'精度':>9} | {'理论值':>12} | {'实测波动范围':>14} | 噪声/信号")
print("-" * 58)
for name, c, t in [("float32", c32, t32), ("float64", c64, t64)]:
    spread = max(c) - min(c)
    print(f"{name:>9} | {t:>12.3e} | {spread:>14.3e} | {spread/t:>8.1e} 倍")

print("\n→ float64 下波动降到 ~1e-14（机器精度），曲线变成一条常数直线。")
print("  同一段数学、同样的输入，仅仅换个精度结论就变了 —— 说明原曲线来自舍入误差。")

In [ ]:
# ══════════ 证据 2：抵消发生在哪里 ══════════
pi  = torch.tensor(0.1).log()
old = torch.tensor(0.005).log()
ref = torch.tensor(0.101).log()
rewards = torch.tensor([1., 0, 0, 0, 0, 0, 0, 0])

l = minimal_grpo_loss(pi, old, ref, rewards, is_debug=False)
true_sum = 8 * 0.01 * grpo_kl(pi, ref)

print("逐样本 loss:")
print(f"  {[round(v, 4) for v in l.tolist()]}")
print(f"\n  单项最大绝对值 : {l.abs().max():.4f}")
print(f"  真实的和       : {true_sum:.4e}")
print(f"  量级比         : {(l.abs().max() / true_sum):.2e} 倍")
print(f"\n  float32 有效数字 ~7 位，而信号比噪声源小 7 个数量级")
print(f"  → 求和时真实信号恰好落在舍入误差以下，被完全淹没")

# 直观演示灾难性抵消
big = torch.tensor(49.4961)
print(f"\n灾难性抵消最小复现：")
print(f"  float32: (49.4961 - 49.4961) + 4e-6 = {((big - big) + 4e-6).item():.4e}  ← 正常")
print(f"  float32: (49.4961 + 4e-6) - 49.4961 = {((big + 4e-6) - big).item():.4e}  ← 信号丢失")
print(f"  同样的数学，不同的求和顺序，结果差异巨大。")

In [ ]:
# ══════════ 证据 3：可视化 float32 vs float64 ══════════
n = 128
c32, t32 = loss_sum_curve(torch.float32, n=n)
c64, t64 = loss_sum_curve(torch.float64, n=n)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].plot(c32, lw=1, color='crimson')
axes[0].axhline(t32, color='k', ls='--', label=f'theoretical G·β·KL = {t32:.2e}')
axes[0].set_title('float32 — looks like a trend, but it is rounding noise')
axes[0].set_xlabel('number of correct answers in group')
axes[0].set_ylabel('loss.sum()')
axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(c64, lw=1, color='steelblue')
axes[1].axhline(t64, color='k', ls='--', label=f'theoretical G·β·KL = {t64:.2e}')
axes[1].set_title('float64 — flat line, exactly as theory predicts')
axes[1].set_xlabel('number of correct answers in group')
axes[1].set_ylabel('loss.sum()')
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

print(f"float32 波动范围: {max(c32)-min(c32):.3e}   (理论值 {t32:.3e})")
print(f"float64 波动范围: {max(c64)-min(c64):.3e}   (理论值 {t64:.3e})")
print("\n左图那条起伏的曲线没有任何训练学意义 —— 它是 float32 的舍入误差。")

---
## 4. 那真实训练中的 loss 呢？

上面的抵消是**玩具设定的产物**：所有样本共用同一个标量 ratio，导致
$\sum_i r \hat{A}_i = r \sum_i \hat{A}_i = 0$ 精确成立。

**真实训练中每条响应有各自的 ratio $r_i$**（不同 token、不同长度、不同的策略更新幅度），
于是策略梯度项变成：

$$\sum_i r_i \hat{A}_i \;\propto\; \text{Cov}(r, \hat{A}) \neq 0$$

这一项**才是真正携带梯度信号的部分**。它的含义是：
「策略更新的方向，是否与优势的方向一致」。

In [ ]:
# ══════════ 标量 ratio vs 逐样本 ratio ══════════
torch.manual_seed(0)
A = grpo_advantage(torch.tensor([1., 1, 1, 0, 0, 0, 0, 0]))
print(f"优势 A = {[round(v,4) for v in A.tolist()]}")
print(f"sum(A) = {A.sum():.3e}\n")

print("情形一：全组共用一个标量 ratio（本 notebook 的玩具设定）")
for r in [0.8, 1.0, 1.5, 20.0]:
    s = (r * A).sum().item()
    note = "= 0（数学上精确为 0）" if s == 0 else f"≈ 0（float32 残差，理论值 0）"
    print(f"  ratio = {r:>5.1f}  →  sum(ratio · A) = {s:+.3e}   {note}")

print("\n情形二：每条响应有各自的 ratio（真实训练）")
for trial in range(4):
    r_i = torch.rand(8) * 0.6 + 0.7          # ratio 在 [0.7, 1.3] 随机
    cov = (r_i * A).sum()
    sign = "策略更新与优势同向 ✅" if cov > 0 else "策略更新与优势反向 ⚠️"
    print(f"  第{trial+1}次采样   →  sum(ratio_i · A_i) = {cov:+.4f}   {sign}")

print("\n→ 只有逐样本 ratio 不同时，这一项才非零、才携带训练信号。")
print("  但即便如此，loss 的绝对值仍然依赖 beta、KL、ratio 分布等多个量，")
print("  单看它的数值仍然无法判断「模型是不是在变好」。")

---
## 5. 该监控什么指标？

既然 loss 不可靠，实际训练中应该盯这几个：

| 指标 | 含义 | 健康表现 |
|------|------|----------|
| **reward / accuracy** | 组内平均奖励 | **稳定上升** ← 这才是真正的训练进展 |
| **KL** | 与参考模型的距离 | 缓慢上升后趋于平稳；暴涨说明策略跑飞 |
| **clip fraction** | 被裁剪的 token 比例 | < 10%；过高说明学习率太大或数据太 off-policy |
| **entropy** | 策略的熵 | 缓慢下降；骤降 = 熵坍缩，多样性丧失 |
| **grad norm** | 梯度范数 | 平稳；尖峰对应不稳定的批次 |
| **有效样本比例** | 组内非全对/全错的比例 | 越高越好；过低说明任务难度不匹配 |
| ~~loss~~ | ~~损失值~~ | ~~无参考价值，在 0 附近震荡属正常~~ |

**为什么 reward 才是正确的指标**：GRPO 的优化目标是最大化期望奖励，
loss 只是为了让自动微分能算梯度而构造的**替代目标（surrogate objective）**。
替代目标的数值大小与真实目标的进展没有单调对应关系。

> 类比：SGD 优化 $f(x)$ 时你看 $f(x)$ 的值；但 GRPO 的 loss 更像是「梯度的势函数」，
> 它的值随每批数据的采样分布漂移，不构成可比较的序列。

In [ ]:
# ══════════ 模拟：训练变好时 reward 单调上升，loss 却在 0 附近震荡 ══════════
torch.manual_seed(42)
G, steps, beta = 8, 60, 0.01

reward_hist, loss_hist, kl_hist = [], [], []
for step in range(steps):
    # 模拟训练进展：正确率从 20% 线性提升到 90%
    acc = 0.2 + 0.7 * step / steps
    rewards = (torch.rand(G) < acc).float()

    # 模拟策略逐渐偏离参考模型：KL 缓慢增大
    pi_lp  = torch.log(torch.rand(G) * 0.3 + 0.1)
    ref_lp = pi_lp - torch.randn(G) * (0.01 + 0.04 * step / steps)
    old_lp = pi_lp - torch.randn(G) * 0.05           # 逐样本 ratio，贴近真实训练

    loss = minimal_grpo_loss(pi_lp, old_lp, ref_lp, rewards, beta=beta, is_debug=False)
    reward_hist.append(rewards.mean().item())
    loss_hist.append(loss.mean().item())
    kl_hist.append(grpo_kl(pi_lp, ref_lp).mean().item())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, data, title, color in [
    (axes[0], reward_hist, 'Reward (mean)  ← WATCH THIS', 'green'),
    (axes[1], loss_hist,   'Loss  ← noisy, no clear trend', 'crimson'),
    (axes[2], kl_hist,     'KL divergence  ← watch for blowup', 'steelblue')]:
    ax.plot(data, color=color, alpha=.55, lw=1)
    k = 9
    smooth = [sum(data[max(0, i-k):i+1]) / len(data[max(0, i-k):i+1]) for i in range(len(data))]
    ax.plot(smooth, color=color, lw=2.5, label='moving average')
    ax.set_title(title); ax.set_xlabel('training step'); ax.legend(); ax.grid(alpha=.3)
axes[1].axhline(0, color='k', ls='--', alpha=.5)
plt.tight_layout(); plt.show()

import statistics
print(f"reward: 前10步均值 {statistics.mean(reward_hist[:10]):.3f} "
      f"→ 后10步均值 {statistics.mean(reward_hist[-10:]):.3f}   ✅ 明确上升")
print(f"loss  : 前10步均值 {statistics.mean(loss_hist[:10]):+.4f} "
      f"→ 后10步均值 {statistics.mean(loss_hist[-10:]):+.4f}   ⚠️ 在 0 附近震荡，无明确趋势")
print(f"KL    : 前10步均值 {statistics.mean(kl_hist[:10]):.5f} "
      f"→ 后10步均值 {statistics.mean(kl_hist[-10:]):.5f}   ✅ 缓慢上升，符合预期")
print("\n→ 模型明明在变好（reward 从 0.2 涨到 0.9），loss 却完全看不出来。")

---
## 6. 总结

### 回答开篇的三个问题

**Q1：为什么 loss 是负数？**
GRPO 的 loss 是「要最大化的目标」取负号，不是交叉熵，没有下界 0。
单样本 loss 的符号由 $\hat{A}_i$ 决定：优势为正 → loss 为负。
由于 $\sum_i \hat{A}_i \equiv 0$，**每组必然同时含正负 loss**。这是设计使然。

**Q2：为什么 loss 会「上升」？**
两种情况要分清：
- **玩具设定下**（全组共用标量 ratio）：loss 的和恒等于 $G\beta\text{KL}$，与奖励无关。
  观察到的「上升」是 float32 灾难性抵消产生的噪声——切到 float64 就变平了。
- **真实训练中**：loss 随 KL 增大而缓慢上升是**正常的**。
  策略在优化奖励的同时必然偏离参考模型，$\beta \text{KL}$ 项随之增大。
  只要 reward 在涨、KL 没有暴涨，loss 上升不是问题。

**Q3：loss 能反映训练进展吗？**
**不能。** 它是替代目标（surrogate objective），数值随每批数据的采样分布漂移。
请监控 **reward / KL / clip fraction / entropy**。

### 数值实践要点

| 要点 | 说明 |
|------|------|
| KL 必须在 log 空间算 | `exp(ref - pi)` 而非 `ref.exp() / pi.exp()`，后者在 log prob 很负时溢出为 `nan` |
| 警惕灾难性抵消 | 大量正负项求和时，float32 的 ~7 位有效数字可能完全淹没真实信号 |
| 用 `mean` 而非 `sum` | 求和会放大抵消问题；取均值同时也让 loss 与组大小 $G$ 解耦 |
| 存疑时切 float64 复核 | 若结论随精度改变，说明它来自舍入误差而非数学性质 |

### 延伸阅读

- `grpo_loss.ipynb` —— GRPO 完整实现（含 clip / mask / shift 对齐）与逐项推导
- `../dapo/dapo_loss.ipynb` —— 解决「组内全对/全错无梯度」与「熵坍缩」
- `../gspo/gspo_loss.ipynb` —— 序列级重要性采样，降低 token 级 IS 的方差